# Screening quantity: controlled setting and natural images

**Computes:** Mutual-information ceiling and the gain of a footprint built with a second source, in a controlled setting and on BSDS500.

**Inputs:** BSDS500 (downloaded by the notebook).  
**Outputs:** `results/03/results_03.json`, intermediate files `results_*.json` of its stages, and a text report.


In [ ]:
import os, json, time
import numpy as np
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")
RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.abspath(os.path.join(RESULTS_DIR, "03"))
os.makedirs(os.path.join(OUTDIR, "reports"), exist_ok=True)
print(f"output dir: {OUTDIR}   run stamp: {RUN_STAMP}")

In [ ]:
import numpy as np
import json, time
R = {"notebook": "03", "started": time.strftime("%Y-%m-%d %H:%M:%S")}

rng = np.random.default_rng(0)
NB = 12
N = 200_000

def _disc(v, nb=NB):
    q = np.quantile(v, np.linspace(0, 1, nb + 1)[1:-1])
    return np.searchsorted(q, v)

def _H(*cols):
    idx = np.zeros(len(cols[0]), dtype=np.int64)
    for c in cols:
        idx = idx * (c.max() + 1) + c
    cnt = np.bincount(idx)
    p = cnt[cnt > 0] / cnt.sum()
    return float(-(p * np.log(p)).sum())

def mi(y, x):
    return _H(y) + _H(x) - _H(y, x)

def cmi(y, s, u):
    return _H(y, u) + _H(s, u) - _H(y, s, u) - _H(u)

def cmi_with_null(y, s, u, n_perm=200, seed=0):
    obs = cmi(y, s, u)
    r = np.random.default_rng(seed)
    null = []
    order = np.argsort(u, kind="stable")
    bounds = np.searchsorted(u[order], np.arange(u.max() + 2))
    for _ in range(n_perm):
        sp = s.copy()
        for a, b in zip(bounds[:-1], bounds[1:]):
            if b - a > 1:
                blk = order[a:b]
                sp[blk] = s[r.permutation(blk)]
        null.append(cmi(y, sp, u))
    null = np.array(null)
    return obs, float(np.median(null)), float((null >= obs).mean()), obs - float(np.median(null))

In [ ]:
z = rng.normal(size=N)
u_c = 1 / (1 + np.exp(-(1.2 * z + 0.9 * rng.normal(size=N))))
y_c = 0.8 * z + 0.6 * rng.normal(size=N)

s_noise = np.clip(u_c + 0.05 * rng.normal(size=N), 0, 1)
s_redundant = u_c ** 2
s_useful = 0.9 * z + 0.7 * rng.normal(size=N)

Y = _disc(y_c)
U = _disc(u_c)

print("=" * 78)
print("A. Controlled setting: three candidate second sources")
print("=" * 78)
print(f"pointwise ceiling  I(Y;u) = {mi(Y, U):.5f} nats")
print()
print(f"{'source':14s} {'I(Y;s)':>9s} {'I(Y;s|u) raw':>13s} {'null median':>12s} "
      f"{'excess':>9s} {'p':>7s} {'headroom':>9s}")
print("-" * 78)
base = mi(Y, U)
res = {}
for nm, s in (("noisy copy", s_noise), ("function of u", s_redundant),
              ("second view", s_useful)):
    S = _disc(s)
    obs, nul, p, exc = cmi_with_null(Y, S, U, n_perm=120, seed=1)
    res[nm] = dict(mi_s=mi(Y, S), cmi_raw=obs, null=nul, excess=exc, p=p,
                   headroom=exc / base)
    print(f"{nm:14s} {mi(Y,S):9.5f} {obs:13.5f} {nul:12.5f} {exc:9.5f} "
          f"{p:7.3f} {exc/base:9.3f}")
print("-" * 78)
print()
for nm, r in res.items():
    verdict = "below 0.05: cannot exceed the ceiling" if r["headroom"] < 0.05 \
        else "0.05 or more: may exceed the ceiling"
    print(f"  {nm:14s} headroom {r['headroom']:6.3f}   {verdict}")

In [ ]:
print()
print("=" * 78)
print("B. Footprints built from each source")
print("=" * 78)

def footprint_pointwise(u):
    return np.abs(u ** 0.85 - u ** 1.15)

def footprint_simple(u, s):
    s01 = (s - s.min()) / max(float(np.ptp(s)), 1e-12)
    return np.abs(u - s01) * 0.5 + footprint_pointwise(u) * 0.5

def footprint_regression(u, s, y, nb=NB):
    U_, S_ = _disc(u, nb), _disc(s, nb)
    key = U_ * (S_.max() + 1) + S_
    num = np.bincount(key, weights=y, minlength=key.max() + 1)
    den = np.bincount(key, minlength=key.max() + 1)
    tab = np.divide(num, np.maximum(den, 1))
    return tab[key]

w_point = footprint_pointwise(u_c)
print(f"{'construction':26s} {'I(Y;w)':>9s} {'vs ceiling':>11s} {'bound':>9s} {'ok':>4s}")
print("-" * 78)
_mp = mi(Y, _disc(w_point))
print(f"{'pointwise f(u)':26s} {_mp:9.5f} {_mp/base:11.3f} {base:9.5f}  yes")
for nm, s in (("noisy copy", s_noise), ("function of u", s_redundant),
              ("second view", s_useful)):
    bound = base + res[nm]["cmi_raw"]
    for lbl, w in (("simple", footprint_simple(u_c, s)),
                   ("regression", footprint_regression(u_c, s, y_c))):
        m = mi(Y, _disc(w))
        print(f"{lbl + ', ' + nm:26s} {m:9.5f} {m/base:11.3f} {bound:9.5f} "
              f"{'yes' if m <= bound + 1e-9 else 'NO':>4s}")
print("-" * 78)
print()
print()

R["ceiling"] = base
R["screening"] = {}
for nm, lbl, sv in (("noisy copy", "noisy copy of u", s_noise),
                    ("function of u", "function of u", s_redundant),
                    ("second view", "independent second view", s_useful)):
    R["screening"][lbl] = dict(
        headroom=res[nm]["headroom"], p=res[nm]["p"],
        simple_h=mi(Y, _disc(footprint_simple(u_c, sv))) / base,
        best_h=mi(Y, _disc(footprint_regression(u_c, sv, y_c))) / base)

In [ ]:
import numpy as np

def _rank01(x):
    r = np.argsort(np.argsort(x)).astype(float)
    return r / max(len(r) - 1, 1)

def candidates(u, s):
    ur, sr = _rank01(u), _rank01(s)
    return {
        "|u - s|": np.abs(ur - sr),
        "|u - s| x boundary": np.abs(ur - sr) * (1.0 - 2.0 * np.abs(ur - 0.5)),
        "sqrt(4 u(1-u) . 4 s(1-s))": np.sqrt(
            np.clip(4 * ur * (1 - ur), 0, 1) * np.clip(4 * sr * (1 - sr), 0, 1)),
        "s alone": sr,
    }

print("=" * 78)
print("C. Candidate constructions from u and s")
print("=" * 78)
base = mi(Y, U)
print(f"pointwise ceiling  I(Y;u) = {base:.5f} nats")
print(f"{'source':14s} {'construction':28s} {'I(Y;w)/I(Y;u)':>14s} {'verdict':>12s}")
print("-" * 78)

SPEND = {}
for nm, s in (("noisy copy", s_noise), ("function of u", s_redundant),
              ("second view", s_useful)):
    row = {}
    for cname, w in candidates(u_c, s).items():
        frac = mi(Y, _disc(w)) / base
        row[cname] = float(frac)
        verdict = ("above 1" if frac > 1.0
                   else "0.25 to 1" if frac > 0.25 else "0.25 or less")
        print(f"{nm:14s} {cname:28s} {frac:14.3f} {verdict:>12s}")
    SPEND[nm] = row
    print("-" * 78)

print()
print(f"{'source':14s} {'simple':>8s} {'best constructible':>20s} {'oracle':>8s}")
for nm in SPEND:
    key = {"noisy copy": "noisy copy of u", "function of u": "function of u",
           "second view": "independent second view"}[nm]
    best = max(SPEND[nm].values())
    print(f"{nm:14s} {R['screening'][key]['simple_h']:8.3f} {best:20.3f} "
          f"{R['screening'][key]['best_h']:8.3f}")

payload = {"ceiling_nats": float(base), "constructible": SPEND,
           "reference": {k: {"simple": R["screening"][k]["simple_h"],
                             "oracle": R["screening"][k]["best_h"],
                             "headroom": R["screening"][k]["headroom"]}
                         for k in R["screening"]}}
with open(os.path.join(OUTDIR, "results_spend.json"), "w") as f:
    json.dump(payload, f, indent=1)

In [ ]:
import numpy as np

def fit_h(u_tr, s_tr, y_tr, u_te, s_te, bins=12):
    qu = np.quantile(u_tr, np.linspace(0, 1, bins + 1)); qu[0] -= 1e-9
    qs = np.quantile(s_tr, np.linspace(0, 1, bins + 1)); qs[0] -= 1e-9
    iu_tr = np.clip(np.searchsorted(qu, u_tr) - 1, 0, bins - 1)
    is_tr = np.clip(np.searchsorted(qs, s_tr) - 1, 0, bins - 1)
    tot = np.zeros((bins, bins)); cnt = np.zeros((bins, bins))
    np.add.at(tot, (iu_tr, is_tr), y_tr)
    np.add.at(cnt, (iu_tr, is_tr), 1.0)
    grid = np.where(cnt > 0, tot / np.maximum(cnt, 1), y_tr.mean())
    iu_te = np.clip(np.searchsorted(qu, u_te) - 1, 0, bins - 1)
    is_te = np.clip(np.searchsorted(qs, s_te) - 1, 0, bins - 1)
    return grid[iu_te, is_te]

print("=" * 78)
print("D. Fitted footprint on a held-out half")
print("=" * 78)

rng_split = np.random.default_rng(7)
idx = rng_split.permutation(len(u_c))
tr, te = idx[: len(idx) // 2], idx[len(idx) // 2:]
base_te = mi(Y[te], _disc(u_c[te]))
print(f"ceiling on the held-out half  I(Y;u) = {base_te:.5f} nats")
print()
print(f"{'source':16s} {'fitted h':>10s} {'u alone':>9s} {'s alone':>9s} "
      f"{'oracle':>8s} {'headroom':>9s}")
print("-" * 78)

FIT = {}
for nm, s, key in (("noisy copy", s_noise, "noisy copy of u"),
                   ("function of u", s_redundant, "function of u"),
                   ("second view", s_useful, "independent second view")):
    w_te = fit_h(u_c[tr], s[tr], y_c[tr], u_c[te], s[te])
    frac = mi(Y[te], _disc(w_te)) / base_te
    fu = mi(Y[te], _disc(u_c[te])) / base_te
    fs = mi(Y[te], _disc(s[te])) / base_te
    FIT[nm] = dict(fitted=float(frac), u_alone=float(fu), s_alone=float(fs),
                   oracle=R["screening"][key]["best_h"],
                   headroom=R["screening"][key]["headroom"])
    print(f"{nm:16s} {frac:10.3f} {fu:9.3f} {fs:9.3f} "
          f"{R['screening'][key]['best_h']:8.3f} "
          f"{R['screening'][key]['headroom']:9.3f}")
print("-" * 78)

print()

payload = {"ceiling_heldout_nats": float(base_te), "fitted": FIT,
           "bins": 12, "split": "half, permuted, seed 7"}
with open(os.path.join(OUTDIR, "results_fit.json"), "w") as f:
    json.dump(payload, f, indent=1)

In [ ]:
import subprocess, glob

ROOT = os.environ.get("BSDS_ROOT") or ("/content/bsds" if IN_COLAB else os.path.abspath(os.path.join(OUTDIR, "bsds")))
BSDS_ROOT = os.path.join(ROOT, "repo", "BSDS500", "data")

if not os.path.isdir(BSDS_ROOT):
    os.makedirs(ROOT, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "-q",
                    "https://github.com/BIDS/BSDS500.git",
                    os.path.join(ROOT, "repo")], check=True)

n_img = len(glob.glob(os.path.join(BSDS_ROOT, "images", "test", "*.jpg")))
n_gt = len(glob.glob(os.path.join(BSDS_ROOT, "groundTruth", "test", "*.mat")))
print(f"test images {n_img} | ground truth {n_gt}")
assert n_img > 0 and n_gt > 0, "BSDS500 not found where expected"

In [ ]:
import os, glob, json
import numpy as np
from scipy.io import loadmat
from scipy.ndimage import sobel, uniform_filter
from PIL import Image

def fcm_membership(g, m=2.0, iters=40):
    v = np.array([np.quantile(g, 0.25), np.quantile(g, 0.75)])
    for _ in range(iters):
        d = np.abs(g[None, :] - v[:, None]) + 1e-12
        w = d ** (-2.0 / (m - 1.0))
        u = w / w.sum(0, keepdims=True)
        um = u ** m
        v = (um * g[None, :]).sum(1) / um.sum(1)
    return u[int(np.argmax(v))]

def gradient_signal(img):
    gx, gy = sobel(img, axis=0), sobel(img, axis=1)
    return uniform_filter(np.hypot(gx, gy), size=5)

def load_split(split, n_images, min_ann=4):
    files = sorted(glob.glob(os.path.join(BSDS_ROOT, "images", split, "*.jpg")))
    out = []
    for f in files:
        if len(out) >= n_images:
            break
        p = os.path.join(BSDS_ROOT, "groundTruth", split,
                         os.path.basename(f)[:-4] + ".mat")
        if not os.path.exists(p):
            continue
        gt = loadmat(p)["groundTruth"][0]
        masks = [np.asarray(g["Boundaries"][0, 0], dtype=float) for g in gt]
        if len(masks) < min_ann:
            continue
        frac = np.stack(masks).mean(0)
        Yi = ((frac > 0) & (frac < 1)).ravel()
        if Yi.sum() < 500 or (~Yi).sum() < 500:
            continue
        img = np.asarray(Image.open(f).convert("L"), dtype=float) / 255.0
        out.append(dict(name=os.path.basename(f), u=fcm_membership(img.ravel()),
                        s=gradient_signal(img).ravel(), Y=Yi))
    return out

def mi_excess(y, x, n_perm=40, seed=0):
    rng = np.random.default_rng(seed)
    obs = mi(_disc(y), _disc(x))
    nul = float(np.median([mi(_disc(rng.permutation(y)), _disc(x))
                           for _ in range(n_perm)]))
    return obs - nul, obs, nul

BINS = 8

def fit_h(u_tr, s_tr, y_tr, u_te, s_te, bins=BINS):
    qu = np.quantile(u_tr, np.linspace(0, 1, bins + 1)); qu[0] -= 1e-9
    qs = np.quantile(s_tr, np.linspace(0, 1, bins + 1)); qs[0] -= 1e-9
    iu = np.clip(np.searchsorted(qu, u_tr) - 1, 0, bins - 1)
    isx = np.clip(np.searchsorted(qs, s_tr) - 1, 0, bins - 1)
    tot = np.zeros((bins, bins)); cnt = np.zeros((bins, bins))
    np.add.at(tot, (iu, isx), y_tr); np.add.at(cnt, (iu, isx), 1.0)
    grid = np.where(cnt > 0, tot / np.maximum(cnt, 1), y_tr.mean())
    return grid[np.clip(np.searchsorted(qu, u_te) - 1, 0, bins - 1),
                np.clip(np.searchsorted(qs, s_te) - 1, 0, bins - 1)]

def pool(rs, cap=400_000, seed=3):
    rng = np.random.default_rng(seed)
    u = np.concatenate([r["u"] for r in rs])
    s = np.concatenate([r["s"] for r in rs])
    y = np.concatenate([r["Y"] for r in rs]).astype(float)
    if len(u) > cap:
        k = rng.choice(len(u), cap, replace=False)
        u, s, y = u[k], s[k], y[k]
    return u, s, y

N_PER_SPLIT = 60
print("=" * 78)
print("E. BSDS500: gradient as the second source")
print("=" * 78)
TR, TE = load_split("train", N_PER_SPLIT), load_split("test", N_PER_SPLIT)
print(f"train images {len(TR)} | test images {len(TE)}")

u_tr, s_tr, y_tr = pool(TR)
u_te, s_te, y_te = pool(TE)
print(f"pixels: train {len(u_tr):,} | test {len(u_te):,} | "
      f"disagreement rate {y_te.mean():.3f}")
print()

w_fit = fit_h(u_tr, s_tr, y_tr, u_te, s_te)
w_u_only = fit_h(u_tr, np.zeros_like(s_tr), y_tr, u_te, np.zeros_like(s_te))

print(f"{'quantity':26s} {'MI (nats)':>11s} {'null':>11s} {'excess':>11s}")
print("-" * 78)
ROWS = {}
for k, (nm, x) in enumerate((("u  (the ceiling)", u_te),
                             ("s  (gradient alone)", s_te),
                             ("fitted h(u) only", w_u_only),
                             ("fitted h(u,s)", w_fit))):
    exc, obs, nul = mi_excess(y_te, x, seed=k)
    ROWS[nm] = dict(excess=float(exc), mi=float(obs), null=float(nul))
    print(f"{nm:26s} {obs:11.6f} {nul:11.6f} {exc:11.6f}")
print("-" * 78)

gain = ROWS["fitted h(u,s)"]["excess"] - ROWS["s  (gradient alone)"]["excess"]
print(f"\ngain of h(u,s) over the gradient alone: {gain:+.6f} nats")

rng = np.random.default_rng(19)
diffs = []
for b in range(20):
    pick = rng.choice(len(TE), len(TE), replace=True)
    ub, sb, yb = pool([TE[i] for i in pick], cap=200_000, seed=100 + b)
    wb = fit_h(u_tr, s_tr, y_tr, ub, sb)
    d = (mi_excess(yb, wb, n_perm=12, seed=b)[0]
         - mi_excess(yb, sb, n_perm=12, seed=b)[0])
    diffs.append(d)
diffs = np.array(diffs)
lo, hi = (float(x) for x in np.percentile(diffs, [2.5, 97.5]))
print(f"image-level bootstrap, 20 resamples: median {np.median(diffs):+.6f}, "
      f"95% interval [{lo:+.6f}, {hi:+.6f}]")

print()
for bn in (4, 6, 8, 12):
    wb = fit_h(u_tr, s_tr, y_tr, u_te, s_te, bins=bn)
    gb = (mi_excess(y_te, wb, n_perm=15, seed=bn)[0]
          - mi_excess(y_te, s_te, n_perm=15, seed=bn)[0])
    print(f"  bins {bn:2d}: {gb:+.5f} nats "
          f"({gb / max(ROWS['s  (gradient alone)']['excess'], 1e-12) * 100:+5.1f}% of what s carries)")

payload = dict(n_train=len(TR), n_test=len(TE), bins=BINS, rows=ROWS,
               gain_nats=float(gain), boot_median=float(np.median(diffs)),
               boot_lo=lo, boot_hi=hi)
with open(os.path.join(OUTDIR, "results_bsds_combine.json"), "w") as f:
    json.dump(payload, f, indent=1)

In [ ]:
u_tr, s_tr, y_tr = pool(TR)
u_te, s_te, y_te = pool(TE)
_, cnull, cp, cexc = cmi_with_null(_disc(y_tr), _disc(s_tr), _disc(u_tr), n_perm=40, seed=3)
base_raw = mi(_disc(y_tr), _disc(u_tr))
base_exc, _, _ = mi_excess(y_tr, u_tr)
ceiling_te, _, _ = mi_excess(y_te, u_te)
ETA = dict(eta=float(cexc / max(base_raw, 1e-12)), eta_excess_denominator=float(cexc / max(base_exc, 1e-12)),
           cmi_excess=float(cexc), cmi_perm_p=float(cp), ceiling_train_raw=float(base_raw),
           ceiling_train_excess=float(base_exc), ceiling_test_excess=float(ceiling_te),
           n_train=len(TR), n_test=len(TE))
print(f"BSDS500: eta = {ETA['eta']:.2f} (excess denominator: {ETA['eta_excess_denominator']:.2f}); "
      f"I(Y;s|u) excess {cexc:.5f} nats, permutation p = {cp:.3f}; I(Y;u) train {base_raw:.5f} raw / {base_exc:.5f} excess")
with open(os.path.join(OUTDIR, "results_bsds_eta.json"), "w") as f:
    json.dump(ETA, f, indent=1)

In [ ]:
import os, json
import numpy as np

N = 200_000
BINS = 8

def make(alpha, seed=0):
    rng = np.random.default_rng(seed)
    z = rng.normal(size=N)
    u = 1 / (1 + np.exp(-(alpha * z + 0.9 * rng.normal(size=N))))
    y = 0.8 * z + 0.6 * rng.normal(size=N)
    s = 0.9 * z + 0.7 * rng.normal(size=N)
    return u, s, y

def excess(y, x, n_perm=25, seed=0):
    rng = np.random.default_rng(seed)
    obs = mi(_disc(y), _disc(x))
    nul = float(np.median([mi(_disc(rng.permutation(y)), _disc(x))
                           for _ in range(n_perm)]))
    return obs - nul

def grid_fit(u_tr, s_tr, y_tr, u_te, s_te, bins=BINS):
    qu = np.quantile(u_tr, np.linspace(0, 1, bins + 1)); qu[0] -= 1e-9
    qs = np.quantile(s_tr, np.linspace(0, 1, bins + 1)); qs[0] -= 1e-9
    iu = np.clip(np.searchsorted(qu, u_tr) - 1, 0, bins - 1)
    isx = np.clip(np.searchsorted(qs, s_tr) - 1, 0, bins - 1)
    tot = np.zeros((bins, bins)); cnt = np.zeros((bins, bins))
    np.add.at(tot, (iu, isx), y_tr); np.add.at(cnt, (iu, isx), 1.0)
    g = np.where(cnt > 0, tot / np.maximum(cnt, 1), y_tr.mean())
    return g[np.clip(np.searchsorted(qu, u_te) - 1, 0, bins - 1),
             np.clip(np.searchsorted(qs, s_te) - 1, 0, bins - 1)]

print("=" * 78)
print("F. Controlled sweep of the ceiling I(Y;u)")
print("=" * 78)
print(f"{'alpha':>6s} {'I(Y;u) nats':>12s} {'eta':>9s} {'gain over s':>12s} "
      f"{'gain as % of s':>15s}")
print("-" * 78)

SWEEP = []
for alpha in (1.2, 0.8, 0.5, 0.3, 0.2, 0.1, 0.05, 0.02):
    u, s, y = make(alpha, seed=1)
    half = N // 2
    tr, te = slice(0, half), slice(half, None)
    base = excess(y[te], u[te], seed=2)
    _, _, p, exc = cmi_with_null(_disc(y[tr]), _disc(s[tr]), _disc(u[tr]),
                                 n_perm=40, seed=3)
    base_tr = mi(_disc(y[tr]), _disc(u[tr]))
    eta = exc / max(base_tr, 1e-12)
    w = grid_fit(u[tr], s[tr], y[tr], u[te], s[te])
    gain = excess(y[te], w, seed=4) - excess(y[te], s[te], seed=5)
    s_only = excess(y[te], s[te], seed=6)
    SWEEP.append(dict(alpha=alpha, ceiling=float(base), eta=float(eta),
                      gain=float(gain), s_only=float(s_only)))
    print(f"{alpha:6.2f} {base:12.5f} {eta:9.3f} {gain:+12.5f} "
          f"{gain / max(s_only, 1e-12) * 100:14.1f}%")
print("-" * 78)

usable = [r for r in SWEEP if r["gain"] > 0.02 * r["s_only"]]
if usable:
    cut = min(r["ceiling"] for r in usable)
    print(f"\nsmallest ceiling with a gain above 2 % of what s carries: I(Y;u) = {cut:.5f} nats")
else:
    cut = float("nan")

print()

payload = dict(sweep=SWEEP, cut_nats=float(cut), bins=BINS, n=N)
with open(os.path.join(OUTDIR, "results_precondition.json"), "w") as f:
    json.dump(payload, f, indent=1)

In [ ]:
import os, glob, json
import numpy as np
from scipy.io import loadmat
from scipy.ndimage import sobel, uniform_filter, gaussian_filter
from PIL import Image

CUT = 0.077

def fcm_on(feat, m=2.0, iters=40):
    g = feat.ravel()
    v = np.array([np.quantile(g, 0.25), np.quantile(g, 0.75)])
    for _ in range(iters):
        d = np.abs(g[None, :] - v[:, None]) + 1e-12
        w = d ** (-2.0 / (m - 1.0))
        u = w / w.sum(0, keepdims=True)
        um = u ** m
        v = (um * g[None, :]).sum(1) / um.sum(1)
    return u[int(np.argmax(v))]

def memberships(rgb, gray):
    grad = uniform_filter(np.hypot(sobel(gray, axis=0), sobel(gray, axis=1)), 5)
    m1 = uniform_filter(gray, 9)
    tex = np.sqrt(np.maximum(uniform_filter(gray ** 2, 9) - m1 ** 2, 0))
    chroma = rgb.std(axis=2)
    mismatch = np.abs(gray - gaussian_filter(gray, 3.0))
    return {"intensity": fcm_on(gray), "texture": fcm_on(tex),
            "chroma": fcm_on(chroma), "scale mismatch": fcm_on(mismatch)}, grad.ravel()

def load_split2(split, n_images, min_ann=4):
    files = sorted(glob.glob(os.path.join(BSDS_ROOT, "images", split, "*.jpg")))
    out = []
    for f in files:
        if len(out) >= n_images:
            break
        p = os.path.join(BSDS_ROOT, "groundTruth", split,
                         os.path.basename(f)[:-4] + ".mat")
        if not os.path.exists(p):
            continue
        gt = loadmat(p)["groundTruth"][0]
        masks = [np.asarray(g["Boundaries"][0, 0], dtype=float) for g in gt]
        if len(masks) < min_ann:
            continue
        frac = np.stack(masks).mean(0)
        Yi = ((frac > 0) & (frac < 1)).ravel()
        if Yi.sum() < 500 or (~Yi).sum() < 500:
            continue
        rgb = np.asarray(Image.open(f).convert("RGB"), dtype=float) / 255.0
        gray = rgb.mean(axis=2)
        us, grad = memberships(rgb, gray)
        out.append(dict(us=us, s=grad, Y=Yi))
    return out

N_IMG = 60
print("=" * 78)
print("G. BSDS500: four memberships against the threshold")
print("=" * 78)
TR2, TE2 = load_split2("train", N_IMG), load_split2("test", N_IMG)
print(f"train images {len(TR2)} | test images {len(TE2)}")

def pool2(rs, key, cap=400_000, seed=3):
    rng = np.random.default_rng(seed)
    u = np.concatenate([r["us"][key] for r in rs])
    s = np.concatenate([r["s"] for r in rs])
    y = np.concatenate([r["Y"] for r in rs]).astype(float)
    if len(u) > cap:
        k = rng.choice(len(u), cap, replace=False)
        u, s, y = u[k], s[k], y[k]
    return u, s, y

print()
print(f"{'membership':18s} {'I(Y;u) nats':>12s} {'above line?':>12s} "
      f"{'gain over s':>12s}")
print("-" * 78)
ROWS2 = {}
for key in ("intensity", "texture", "chroma", "scale mismatch"):
    u_tr, s_tr, y_tr = pool2(TR2, key)
    u_te, s_te, y_te = pool2(TE2, key, seed=4)
    ceil = mi_excess(y_te, u_te, n_perm=20, seed=1)[0]
    ok = ceil > CUT
    w = fit_h(u_tr, s_tr, y_tr, u_te, s_te)
    gain = (mi_excess(y_te, w, n_perm=20, seed=2)[0]
            - mi_excess(y_te, s_te, n_perm=20, seed=3)[0])
    ROWS2[key] = dict(ceiling=float(ceil), above_line=bool(ok), gain=float(gain))
    print(f"{key:18s} {ceil:12.5f} {'yes' if ok else 'no':>12s} {gain:+12.5f}")
print("-" * 78)
print(f"the line, from the controlled sweep: I(Y;u) > {CUT:.3f} nats")

s_ref = mi_excess(np.concatenate([r['Y'] for r in TE2]).astype(float)[:400_000],
                  np.concatenate([r['s'] for r in TE2])[:400_000],
                  n_perm=20, seed=9)[0]
print(f"the gradient alone carries {s_ref:.5f} nats")

with open(os.path.join(OUTDIR, "results_memberships.json"), "w") as f:
    json.dump(dict(rows=ROWS2, cut=CUT, gradient_nats=float(s_ref),
                   n_images=len(TE2)), f, indent=1)

In [ ]:
L = lambda n: json.load(open(os.path.join(OUTDIR, n)))
SP, FI, BC, PR, ME, ET = (L("results_spend.json"), L("results_fit.json"), L("results_bsds_combine.json"),
                          L("results_precondition.json"), L("results_memberships.json"), L("results_bsds_eta.json"))
sw = {round(r["alpha"], 2): r for r in PR["sweep"]}
rows = [
    ("controlled: headroom eta of the second view", SP["reference"]["independent second view"]["headroom"], ".3f"),
    ("controlled: fitted h / ceiling (held out)", FI["fitted"]["second view"]["fitted"], ".3f"),
    ("sweep alpha=1.2: ceiling", sw[1.2]["ceiling"], ".3f"),
    ("sweep alpha=1.2: eta", sw[1.2]["eta"], ".2f"),
    ("sweep alpha=1.2: gain as % of s", 100 * sw[1.2]["gain"] / sw[1.2]["s_only"], ".1f"),
    ("sweep alpha=0.5: ceiling (the line)", sw[0.5]["ceiling"], ".3f"),
    ("sweep alpha=0.5: eta", sw[0.5]["eta"], ".2f"),
    ("sweep alpha=0.5: gain as % of s", 100 * sw[0.5]["gain"] / sw[0.5]["s_only"], ".1f"),
    ("sweep alpha=0.2: ceiling", sw[0.2]["ceiling"], ".3f"),
    ("sweep alpha=0.2: eta", sw[0.2]["eta"], ".1f"),
    ("cut (nats)", PR["cut_nats"], ".3f"),
    ("BSDS: I(Y;u) excess, test", BC["rows"]["u  (the ceiling)"]["excess"], ".5f"),
    ("BSDS: I(Y;s) excess, test", BC["rows"]["s  (gradient alone)"]["excess"], ".4f"),
    ("BSDS: gradient / membership", BC["rows"]["s  (gradient alone)"]["excess"] / BC["rows"]["u  (the ceiling)"]["excess"], ".0f"),
    ("BSDS: gain of h(u,s) over s", BC["gain_nats"], "+.4f"),
    ("BSDS: 95% interval, low", BC["boot_lo"], "+.4f"),
    ("BSDS: 95% interval, high", BC["boot_hi"], "+.4f"),
    ("BSDS: eta", ET["eta"], ".1f"),
    ("BSDS: best of four memberships (nats)", max(r["ceiling"] for r in ME["rows"].values()), ".3f"),
]
LINES = [f"03   (run {RUN_STAMP})", f"{'quantity':46s} {'value':>12s}"]
for lab, v, fmt in rows:
    LINES.append(f"{lab:46s} {format(v, fmt):>12s}")
LINES += ["", "memberships: " + ", ".join(f"{k} {r['ceiling']:.5f}" for k, r in ME["rows"].items()),
          f"BSDS eta, excess-denominator variant: {ET['eta_excess_denominator']:.1f}"]
report = "\n".join(LINES) + "\n"; print(report)
payload = dict(notebook="03", run_stamp=RUN_STAMP,
               spend=SP, fit=FI, bsds_combine=BC, precondition=PR, memberships=ME, bsds_eta=ET,
               quoted={lab: v for lab, v, _ in rows})
paths = [os.path.join(OUTDIR, "results_03.json"), os.path.join(OUTDIR, "results_03_report.txt"),
         os.path.join(OUTDIR, "reports", f"results_03_{RUN_STAMP}.json"),
         os.path.join(OUTDIR, "reports", f"results_03_report_{RUN_STAMP}.txt")]
for p in paths:
    with open(p, "w") as fh:
        if p.endswith(".json"): json.dump(payload, fh, indent=1, default=float)
        else: fh.write(report)
        fh.flush(); os.fsync(fh.fileno())
for p in paths:
    ok = os.path.exists(p) and os.path.getsize(p) > 0
    print(f"{'written' if ok else 'NOT WRITTEN'}: {p} ({os.path.getsize(p) if ok else 0} bytes)")